## 1. Importing Required Libraries

In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
#Loading Data
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
print(f"No of rows : {df.shape[0]}")
print(f"No of columns : {df.shape[1]}")

No of rows : 569
No of columns : 33


## Removing unnecessary data 

In [5]:
df.drop(columns=['id', 'Unnamed: 32'], inplace=True)

In [6]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## Train - test split

In [7]:
X_train, x_test, y_train, y_test = train_test_split(
    df.iloc[:, 1:],
    df.iloc[:, 0],
    test_size=0.2,
)

## EDA 

## 1.Scaling 

In [8]:
scaler = StandardScaler()
x_train = scaler.fit_transform(X_train)
x_test = scaler.transform(x_test)

## 2. LabelEncoder

In [9]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

## Numpy array to PyTorch tensors

In [31]:
X_train_tensor = torch.from_numpy(x_train.astype(np.float32))
X_test_tensor = torch.from_numpy(x_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [32]:
print(f"No of rows : {X_train_tensor.shape[0]}")
print(f"No of columns : {X_train_tensor.shape[1]}")

No of rows : 455
No of columns : 30


In [33]:
print(f"No of rows : {y_train_tensor.shape[0]}")

No of rows : 455


## Defining the model

In [34]:
import torch.nn as nn


class MySimpleNN(nn.Module):

  def __init__(self, num_features):

    super().__init__()
    self.linear = nn.Linear(num_features, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, features):

    out = self.linear(features)
    out = self.sigmoid(out)

    return out

## Training Parameters

In [35]:
learning_rate = 0.1
epochs = 25

In [36]:
loss_function = nn.BCELoss()

## Training Pipeline

In [37]:
# create model
model = MySimpleNN(X_train_tensor.shape[1])

# define optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# define loop
for epoch in range(epochs):

  # forward pass
  y_pred = model(X_train_tensor)

  # loss calculate
  loss = loss_function(y_pred, y_train_tensor.view(-1,1))

  # clear gradients
  optimizer.zero_grad()

  # backward pass
  loss.backward()

  # parameters update
  optimizer.step()

  # print loss in each epoch
  print(f'Epoch: {epoch + 1}, Loss: {loss.item()}')

Epoch: 1, Loss: 0.7843297719955444
Epoch: 2, Loss: 0.585326611995697
Epoch: 3, Loss: 0.4822709560394287
Epoch: 4, Loss: 0.41914448142051697
Epoch: 5, Loss: 0.37591344118118286
Epoch: 6, Loss: 0.34410232305526733
Epoch: 7, Loss: 0.31950750946998596
Epoch: 8, Loss: 0.2997962236404419
Epoch: 9, Loss: 0.28356337547302246
Epoch: 10, Loss: 0.26990774273872375
Epoch: 11, Loss: 0.25822246074676514
Epoch: 12, Loss: 0.248082235455513
Epoch: 13, Loss: 0.23917929828166962
Epoch: 14, Loss: 0.23128478229045868
Epoch: 15, Loss: 0.22422461211681366
Epoch: 16, Loss: 0.2178637832403183
Epoch: 17, Loss: 0.21209582686424255
Epoch: 18, Loss: 0.20683540403842926
Epoch: 19, Loss: 0.20201341807842255
Epoch: 20, Loss: 0.19757318496704102
Epoch: 21, Loss: 0.19346767663955688
Epoch: 22, Loss: 0.1896575689315796
Epoch: 23, Loss: 0.18610969185829163
Epoch: 24, Loss: 0.18279583752155304
Epoch: 25, Loss: 0.1796918362379074


In [40]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test_tensor)
  y_pred = (y_pred > 0.5).float()
  accuracy = (y_pred == y_test_tensor).float().mean()
  print(f'Accuracy: {accuracy.item()}')


Accuracy: 0.5221606492996216
